In [0]:
# Read the uploaded CSV into a Spark DataFrame
df_running = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/safetrack_data/etrain_delays.csv")

# Preview it — always check before writing
display(df_running.limit(10))
print(f"Total rows: {df_running.count()}")
print("Columns:", df_running.columns)

In [0]:
# Write to Delta Lake table (this is what makes Databricks usage genuine)
df_running.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("running_history")

print("running_history table created successfully")

# Verify it's queryable
spark.sql("SELECT COUNT(*) as total_records FROM running_history").show()
spark.sql("SELECT * FROM running_history LIMIT 5").show()

In [0]:
# ================================
# 1. IMPORTS
# ================================
from pyspark.sql.functions import col, explode, upper

# ================================
# 2. LOAD JSON
# ================================
df_json = spark.read.json("/Volumes/workspace/default/safetrack_data/stations.json")

# ================================
# 3. FLATTEN JSON
# ================================
df_stations = df_json.select(explode(col("features")).alias("feature"))
df_stations = df_stations.select("feature.properties.*")

print("Raw stations preview:")
display(df_stations.limit(10))

print("Columns:", df_stations.columns)


# ================================
# 4. CLEAN DATA
# ================================
df_stations = df_stations \
    .filter(col("code").isNotNull()) \
    .filter(col("zone").isNotNull()) \
    .filter(col("zone") != "?") \
    .withColumn("code", upper(col("code")))

# Select only useful columns
df_stations = df_stations.select(
    col("code").alias("station_code"),
    col("name").alias("station_name"),
    col("state"),
    col("zone")
)

print("Cleaned stations preview:")
display(df_stations.limit(10))


# ================================
# 5. WRITE TO DELTA TABLE
# ================================
df_stations.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("stations_cleaned")

print("stations_cleaned table created successfully!")


# ================================
# 6. VERIFY (EXACT SAME AS YOUR RUNNING_HISTORY)
# ================================
spark.sql("SELECT COUNT(*) as total_stations FROM stations_cleaned").show()

spark.sql("SELECT * FROM stations_cleaned LIMIT 5").show()